<a href="https://colab.research.google.com/github/Shubhamspc90/GenAI-Using-LangChain/blob/lang/Agent/building%20agent/7__Multi_Tool_Research_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install libraries
!pip install -q langchain langchain-ollama ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 19.6 MB/s eta 0:00:00


In [8]:
from google.colab import userdata

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent

from ddgs import DDGS

ollama_api_key = userdata.get("OLLAMA_API_KEY")
print("API KEY FOUND : " , bool(ollama_api_key))

API KEY FOUND :  True


In [9]:
# creating model
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gpt-oss:20b-cloud",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {ollama_api_key}"
        }
    },
    temperature=0.1
)

In [13]:
# Web Search Tool
from langchain_core.tools import tool
@tool
def web_search(query : str) -> list:
  """ Search the web and return search result"""
  search = DDGS()
  results = search.text(
      query,
      max_result = 5
  )
  return results

# Test Web Search Tool
response = web_search.invoke({
    "query": "Price of latest iphone Duo?"
})
print(response)

[{'title': 'Shop iPhone Duo - Apple', 'href': 'https://www.apple.com/shop/buy-iphone/iphone-duo', 'body': 'Get $35-$885 off a new iPhone Duo when you trade in an iPhone 8 Plus or newer. 0% financing available. Learn more at apple.com.'}, {'title': 'iPhone Duo - Apple', 'href': 'https://www.apple.com/iphone-duo/', 'body': "Introducing iPhone Duo, the first foldable iPhone. When open, the dual display system features the largest, thinnest iPhone display ever. Through reimagined iOS experiences, it offers unparalleled versatility in all-new poses and orientations. All in a pocketable design. Because it's an iPhone, it holds its value longer than other smartphones."}, {'title': 'iPhone Duo (Pre-Order): Prices, Colors, Specs | AT&T', 'href': 'https://www.att.com/buy/phones/apple-iphone-duo.html', 'body': "Pre-order the iPhone Duo at AT&T, Apple's first foldable phone, in Star White or Night Sky. Opens to a 7.6-inch display, with 44 hours of video when folded."}, {'title': 'New iPhone Duo: R

In [22]:
# Discount Tool
@tool
def discount_cal( price : float , discount_rate : float ) -> dict :
    """calculate the discount and final price """
    discount = price * discount_rate / 100
    final_price =  price - discount

    return{
        "Price " : price,
        "Discount Rate " : discount_rate,
        "Discount ": discount ,
        "Final Price": final_price
    }

# Test the Discount Tool
result = discount_cal.invoke({
    "price" : 450000,
    "discount_rate":15
})
print(result)

{'Price ': 450000.0, 'Discount Rate ': 15.0, 'Discount ': 67500.0, 'Final Price': 382500.0}


In [23]:
# GST Tool
@tool
def gst_cal(amount: float, gst_rate: float) -> dict:
    """Calculate GST amount and final price including GST."""

    gst = amount * gst_rate / 100
    total = amount + gst

    return {
        "Amount": amount,
        "GST Rate": gst_rate,
        "GST Amount": gst,
        "Total Amount": total
    }


# Test the GST Tool
result = gst_cal.invoke({
    "amount" : 382500,
    "gst_rate" : 18
})
print(result)


{'Amount': 382500.0, 'GST Rate': 18.0, 'GST Amount': 68850.0, 'Total Amount': 451350.0}


In [24]:
# Create Multi-Tool Agent
agent = create_agent(
    model = llm,
    tools = [ web_search, discount_cal , gst_cal]
)

In [25]:
# Test Multiple Tools
user_question = """
A product costs ₹50,000.
Apply a 10% discount and then apply 18% GST on the discounted price.
What is the final price?
"""

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_question
        }
    ]
})

print(result["messages"][-1].content)

**Final price:** ₹53,100

**Breakdown**

| Step | Calculation | Amount (₹) |
|------|-------------|------------|
| Original price | – | 50,000 |
| 10 % discount | 50,000 × 0.10 | 5,000 |
| Price after discount | 50,000 – 5,000 | 45,000 |
| 18 % GST | 45,000 × 0.18 | 8,100 |
| Final price | 45,000 + 8,100 | **53,100** |

So, after applying a 10 % discount and then adding 18 % GST, the product costs ₹53,100.


In [27]:
# Test Search + Calculation
user_question = """
Search for the current price of a laptop costing around ₹50,000  with name and brand ,
then calculate the price after a 10% discount and 18% GST.
"""

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_question
        }
    ]
})

print(result["messages"][-1].content)

**Laptop selected**

| Brand | Model | Current Price (₹) |
|-------|-------|-------------------|
| **HP** | **HP 15s (15s‑EQ2143AU)** | ₹52,490 (latest price on 24 Sep 2026) |

---

### Price calculations

| Step | Calculation | Result (₹) |
|------|-------------|------------|
| **Original price** | ₹52,490 | 52,490 |
| **10 % discount** | 52,490 × 0.10 = 5,249 | 5,249 |
| **Price after discount** | 52,490 – 5,249 = 47,241 | 47,241 |
| **18 % GST** | 47,241 × 0.18 = 8,503.38 | 8,503.38 |
| **Final price (incl. GST)** | 47,241 + 8,503.38 = 55,744.38 | **₹55,744.38** |

**Result:**  
The HP 15s laptop, priced at ₹52,490, will cost **₹55,744.38** after a 10 % discount and 18 % GST.
